# ACFD-Transformer v3 — Khôi phục toàn bộ số liệu cho bản thảo JOCO
**Dataset:** CICAPT-IIoT phần *network traffic* — `phase1_NetworkData.csv` (5.1 GB) + `phase2_NetworkData.csv` (4.04 GB) tại `/My Drive/ACDF/CIC_APT_2024_Data/`. Yêu cầu **Colab Pro** (High-RAM + GPU).

**Pipeline đúng phương pháp:**
1. Đọc 2 file (chunked, lần đầu ~10–20 phút) → giữ TOÀN BỘ dòng attack + sample benign → **cache .npz trên Drive** (các lần sau load trong vài giây).
2. Mỗi seed: split 70/15/15 → MinMax fit trên train → sliding window (nhãn = dòng cuối window) → ACFD (DDPM **T=1000**, đúng Table 3) chỉ augment **train** → Longformer (HF, 2 layers/8 heads/d=128) → early stopping theo val-F1 → đánh giá trên test **100% dữ liệu thật**.
3. Chạy 3 seeds → **mean ± std thật** cho mọi bảng.
4. **SHAP thật** (GradientExplainer) → thay Fig. 4. **Histogram dữ liệu thật** → thay Fig. 2.
5. Ablation `use_acfd=False` → Table 5 thật.

**Nguyên tắc:** số chạy ra là số đưa vào bản thảo. Không gõ tay, không simulated, không synthetic trong test.


In [ ]:
# ===== Cell 1: Setup & Config =====
!pip install -q transformers shap

from google.colab import drive
drive.mount('/content/drive')

import os, math, json, random, time, gc
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, classification_report, confusion_matrix)
from transformers import LongformerConfig, LongformerModel

class CFG:
    PHASE1   = '/content/drive/My Drive/ACDF/CIC_APT_2024_Data/phase1_NetworkData.csv'
    PHASE2   = '/content/drive/My Drive/ACDF/CIC_APT_2024_Data/phase2_NetworkData.csv'
    OUT_DIR  = '/content/drive/My Drive/ACDF/acfd_v3_outputs'
    CACHE    = '/content/drive/My Drive/ACDF/acfd_v3_outputs/cicapt_network_cache.npz'

    N_FEATURES    = 63          # §4.2
    TARGET_BENIGN = 50_000      # tổng benign sample (từ cả 2 phase)
    CHUNKSIZE     = 1_000_000   # đọc CSV theo chunk

    WINDOW   = 10               # Fig. 3
    # ACFD — Table 3
    T_STEPS  = 1000
    BETA_START, BETA_END = 1e-4, 0.02
    ACFD_HIDDEN, ACFD_EPOCHS, ACFD_LR = 128, 100, 1e-3
    # Longformer — Table 3
    EMBED_DIM, N_LAYERS, N_HEADS, FFN_DIM, DROPOUT = 128, 2, 8, 512, 0.1
    # Training — §3.5
    BATCH_SIZE, LR, MAX_EPOCHS, PATIENCE = 64, 1e-4, 50, 7

    SEEDS = [42, 43, 44]        # multi-seed cho mean ± std

os.makedirs(CFG.OUT_DIR, exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device, '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

In [ ]:
# ===== Cell 2: Load dữ liệu (chunked) + cache =====
META_COLS = {'ts', 'Source IP', 'Destination IP', 'Source Port', 'Destination Port',
             'Protocol_name', 'label', 'subLabel', 'subLabelCat', 'Flow ID', 'id'}

def attack_mask_from(sub):
    s = sub.astype(str).str.strip()
    return (s != '0') & (s != '0.0') & (s != '') & (s.str.lower() != 'nan')

if os.path.exists(CFG.CACHE):
    print('✓ Tìm thấy cache — load trong vài giây...')
    z = np.load(CFG.CACHE, allow_pickle=True)
    X_all, y_all, sub_codes = z['X'], z['y'], z['sub']
    FEATURES, SUB_NAMES = list(z['features']), list(z['sub_names'])
else:
    print('Chưa có cache — đọc 2 file CSV theo chunk (lần đầu ~10–20 phút)...')
    head = pd.read_csv(CFG.PHASE2, nrows=5)
    print(f'Phase2 có {len(head.columns)} cột:', list(head.columns))
    feat_candidates = [c for c in head.columns if c not in META_COLS]

    rng = np.random.default_rng(123)            # seed CỐ ĐỊNH cho bước thu thập dữ liệu
    attack_parts, benign_parts = [], []
    benign_cap = CFG.TARGET_BENIGN * 3          # thu dư rồi subsample chính xác sau

    def harvest(path, phase):
        """Giữ toàn bộ attack; sample benign theo fraction nhỏ mỗi chunk."""
        n_rows = 0
        usecols = feat_candidates + [c for c in ('subLabelCat',) if c in head.columns]
        for chunk in pd.read_csv(path, chunksize=CFG.CHUNKSIZE, usecols=lambda c: c in set(usecols)):
            n_rows += len(chunk)
            if phase == 2 and 'subLabelCat' in chunk.columns:
                m = attack_mask_from(chunk['subLabelCat'])
                if m.any():
                    atk = chunk[m].copy()
                    attack_parts.append(atk)
                ben = chunk[~m]
            else:
                ben = chunk
            # sample benign ~0.5% mỗi chunk, dừng khi đủ cap
            if sum(len(b) for b in benign_parts) < benign_cap and len(ben):
                take = max(1, int(len(ben) * 0.005))
                idx = rng.choice(len(ben), size=min(take, len(ben)), replace=False)
                benign_parts.append(ben.iloc[idx].copy())
            del chunk; gc.collect()
        print(f'   phase{phase}: đã quét {n_rows:,} dòng')

    t0 = time.time()
    harvest(CFG.PHASE2, 2)
    harvest(CFG.PHASE1, 1)
    print(f'Đọc xong sau {(time.time()-t0)/60:.1f} phút')

    attack_df = pd.concat(attack_parts, ignore_index=True)
    benign_df = pd.concat(benign_parts, ignore_index=True)
    if len(benign_df) > CFG.TARGET_BENIGN:
        benign_df = benign_df.sample(n=CFG.TARGET_BENIGN, random_state=123)
    print(f'Attack THẬT: {len(attack_df):,} | Benign sample: {len(benign_df):,}')
    print('Phân bố attack theo subLabelCat:'); print(attack_df['subLabelCat'].value_counts())

    # --- sub-type làm điều kiện ACFD: 0 = benign, 1..K = loại attack ---
    sub_attack, SUB_NAMES = pd.factorize(attack_df['subLabelCat'].astype(str))
    sub_codes = np.concatenate([np.zeros(len(benign_df), dtype=np.int64), sub_attack + 1])
    SUB_NAMES = ['benign'] + list(SUB_NAMES)

    df = pd.concat([benign_df, attack_df], ignore_index=True)
    y_all = np.concatenate([np.zeros(len(benign_df), dtype=np.int64),
                            np.ones(len(attack_df), dtype=np.int64)])

    # --- chọn 63 features: ép numeric, bỏ hằng số, top variance ---
    num = df[[c for c in feat_candidates if c in df.columns]].apply(pd.to_numeric, errors='coerce')
    num = num.replace([np.inf, -np.inf], np.nan).fillna(0)
    num = num.loc[:, num.nunique() > 1]
    print(f'Feature numeric khả dụng: {num.shape[1]}')
    FEATURES = num.var().sort_values(ascending=False).index[:CFG.N_FEATURES].tolist()
    X_all = num[FEATURES].values.astype(np.float32)

    np.savez_compressed(CFG.CACHE, X=X_all, y=y_all, sub=sub_codes,
                        features=np.array(FEATURES), sub_names=np.array(SUB_NAMES, dtype=object))
    print('✓ Đã lưu cache:', CFG.CACHE)
    del df, attack_df, benign_df, num; gc.collect()

print(f'\nDataset THẬT: X={X_all.shape} | benign={(y_all==0).sum():,} | attack={(y_all==1).sum():,}')
print(f'{len(FEATURES)} features (đưa vào Appendix 3):')
for i in range(0, len(FEATURES), 5): print('  ', FEATURES[i:i+5])
print('Sub-types:', SUB_NAMES)

In [ ]:
# ===== Cell 3: Định nghĩa model (ACFD + Longformer) và helpers =====
class SinusoidalTimeEmb(nn.Module):
    def __init__(self, dim): super().__init__(); self.dim = dim
    def forward(self, t):
        half = self.dim // 2
        freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device) / half)
        a = t[:, None].float() * freqs[None]
        return torch.cat([a.sin(), a.cos()], dim=-1)

class ACFDDenoiser(nn.Module):
    """eps_theta(x_t, t, c) — 3-layer MLP 128 units (Table 3), điều kiện sub-type (Eq. 4)."""
    def __init__(self, x_dim, n_cond, t_dim=64, c_dim=64, hidden=CFG.ACFD_HIDDEN):
        super().__init__()
        self.t_emb = SinusoidalTimeEmb(t_dim)
        self.c_emb = nn.Embedding(n_cond, c_dim)
        self.net = nn.Sequential(
            nn.Linear(x_dim + t_dim + c_dim, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, x_dim))
    def forward(self, x, t, c):
        return self.net(torch.cat([x, self.t_emb(t), self.c_emb(c)], dim=-1))

class ACFD:
    """DDPM T=1000, linear beta 1e-4→0.02 (Table 3)."""
    def __init__(self, x_dim, n_cond):
        self.T = CFG.T_STEPS
        self.betas  = torch.linspace(CFG.BETA_START, CFG.BETA_END, self.T, device=device)
        self.alphas = 1.0 - self.betas
        self.abar   = torch.cumprod(self.alphas, dim=0)
        self.model  = ACFDDenoiser(x_dim, n_cond).to(device)
    def pretrain(self, X, c, epochs=CFG.ACFD_EPOCHS, bs=256, lr=CFG.ACFD_LR, verbose=True):
        opt = torch.optim.AdamW(self.model.parameters(), lr=lr)
        dl = DataLoader(TensorDataset(X, c), batch_size=bs, shuffle=True)
        self.model.train()
        for ep in range(epochs):
            tot = 0.0
            for xb, cb in dl:
                xb, cb = xb.to(device), cb.to(device)
                t = torch.randint(0, self.T, (len(xb),), device=device)
                eps = torch.randn_like(xb)
                ab = self.abar[t][:, None]
                xt = ab.sqrt() * xb + (1 - ab).sqrt() * eps
                loss = nn.functional.mse_loss(self.model(xt, t, cb), eps)
                opt.zero_grad(); loss.backward(); opt.step(); tot += loss.item()
            if verbose and (ep + 1) % 20 == 0:
                print(f'      [ACFD] epoch {ep+1}/{epochs} noise-MSE={tot/len(dl):.4f}')
    @torch.no_grad()
    def sample(self, n, cond_pool, x_dim, bs=2048):
        self.model.eval(); out = []
        for s in range(0, n, bs):
            m = min(bs, n - s)
            x = torch.randn(m, x_dim, device=device)
            c = cond_pool[torch.randint(0, len(cond_pool), (m,))].to(device)
            for t in reversed(range(self.T)):
                tt = torch.full((m,), t, dtype=torch.long, device=device)
                eps = self.model(x, tt, c)
                a, ab, b = self.alphas[t], self.abar[t], self.betas[t]
                x = (x - (1 - a) / (1 - ab).sqrt() * eps) / a.sqrt()
                if t > 0: x = x + b.sqrt() * torch.randn_like(x)
            out.append(x.clamp(0, 1).cpu())
        return torch.cat(out)

class LongformerAPTDetector(nn.Module):
    """HF Longformer: 2 layers, 8 heads, d=128 (Table 3). vocab_size=4 để loại word-emb thừa."""
    def __init__(self, input_dim, embed_dim=CFG.EMBED_DIM):
        super().__init__()
        self.embedding = nn.Linear(input_dim, embed_dim)
        cfg = LongformerConfig(
            vocab_size=4, hidden_size=embed_dim, num_hidden_layers=CFG.N_LAYERS,
            num_attention_heads=CFG.N_HEADS, intermediate_size=CFG.FFN_DIM,
            attention_window=[CFG.WINDOW] * CFG.N_LAYERS,   # phủ toàn bộ W=10
            max_position_embeddings=CFG.WINDOW + 8,
            hidden_dropout_prob=CFG.DROPOUT, attention_probs_dropout_prob=CFG.DROPOUT,
            pad_token_id=0, bos_token_id=1, eos_token_id=2, sep_token_id=2)
        self.transformer = LongformerModel(cfg, add_pooling_layer=False)
        self.head = nn.Sequential(nn.Linear(embed_dim, 64), nn.ReLU(),
                                  nn.Dropout(CFG.DROPOUT), nn.Linear(64, 2))
    def forward(self, x):
        h = self.transformer(inputs_embeds=self.embedding(x)).last_hidden_state
        return self.head(h.mean(dim=1))   # GAP + MLP head (Fig. 1)

def make_windows(X, y, sub=None, w=CFG.WINDOW):
    """Window = dòng i..i+w-1; NHÃN = dòng CUỐI window (đã sửa bug lệch nhãn)."""
    xs = np.lib.stride_tricks.sliding_window_view(X, (w, X.shape[1]))[:, 0]
    ys = y[w - 1:]
    out = [torch.tensor(xs.copy(), dtype=torch.float32), torch.tensor(ys, dtype=torch.long)]
    if sub is not None: out.append(torch.tensor(sub[w - 1:], dtype=torch.long))
    return out

@torch.no_grad()
def evaluate(model, dl):
    model.eval(); ys, ps, probs = [], [], []
    for xb, yb in dl:
        logits = model(xb.to(device))
        ys.append(yb); ps.append(logits.argmax(-1).cpu())
        probs.append(logits.softmax(-1)[:, 1].cpu())
    return torch.cat(ys).numpy(), torch.cat(ps).numpy(), torch.cat(probs).numpy()

_tmp = LongformerAPTDetector(len(FEATURES))
N_PARAMS = sum(p.numel() for p in _tmp.parameters() if p.requires_grad)
del _tmp
print(f'Số tham số Longformer detector (đưa vào Table 4): {N_PARAMS:,} ({N_PARAMS/1e6:.2f}M)')

In [ ]:
# ===== Cell 4: run_experiment(seed, use_acfd) =====
def run_experiment(seed, use_acfd=True, keep_artifacts=False):
    set_seed(seed)
    print(f'\n{"="*64}\n  SEED {seed} | ACFD = {use_acfd}\n{"="*64}')

    # --- split 70/15/15 trên DỮ LIỆU THẬT ---
    X_tr, X_tmp, y_tr, y_tmp, s_tr, _ = train_test_split(
        X_all, y_all, sub_codes, test_size=0.30, stratify=y_all, random_state=seed)
    X_va, X_te, y_va, y_te = train_test_split(
        X_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=seed)[:4]

    scaler = MinMaxScaler().fit(X_tr)                      # fit CHỈ trên train
    X_tr, X_va, X_te = scaler.transform(X_tr), scaler.transform(X_va), scaler.transform(X_te)

    Xw_tr, yw_tr, sw_tr = make_windows(X_tr, y_tr, s_tr)
    Xw_va, yw_va = make_windows(X_va, y_va)[:2]
    Xw_te, yw_te = make_windows(X_te, y_te)[:2]
    print(f'Windows: train {tuple(Xw_tr.shape)} | val {tuple(Xw_va.shape)} | test {tuple(Xw_te.shape)}'
          f' | APT-rate train {yw_tr.float().mean():.4f}')

    # --- sanity: dữ liệu phải có tín hiệu học được ---
    clf = LogisticRegression(max_iter=1000, class_weight='balanced')
    clf.fit(Xw_tr[:, -1, :].numpy(), yw_tr.numpy())
    print(f'[Sanity] LogReg val F1 = {f1_score(yw_va.numpy(), clf.predict(Xw_va[:, -1, :].numpy())):.4f}')

    # --- ACFD: augment CHỈ train ---
    if use_acfd:
        x_dim = CFG.WINDOW * len(FEATURES)
        mask = yw_tr == 1
        cond_pool = sw_tr[mask]
        n_cond = len(SUB_NAMES)
        print(f'Pre-train ACFD trên {int(mask.sum())} cửa sổ attack thật, {n_cond} điều kiện...')
        acfd = ACFD(x_dim, n_cond)
        acfd.pretrain(Xw_tr[mask].reshape(-1, x_dim), cond_pool)
        n_need = int((yw_tr == 0).sum() - (yw_tr == 1).sum())
        print(f'Sinh {n_need:,} synthetic (T={CFG.T_STEPS})...')
        t0 = time.time()
        synth = acfd.sample(n_need, cond_pool, x_dim).reshape(-1, CFG.WINDOW, len(FEATURES))
        print(f'   xong sau {(time.time()-t0)/60:.1f} phút')
        Xw_bal = torch.cat([Xw_tr, synth])
        yw_bal = torch.cat([yw_tr, torch.ones(len(synth), dtype=torch.long)])
    else:
        acfd, Xw_bal, yw_bal = None, Xw_tr, yw_tr

    # --- train Longformer ---
    model = LongformerAPTDetector(len(FEATURES)).to(device)
    opt   = torch.optim.AdamW(model.parameters(), lr=CFG.LR)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CFG.MAX_EPOCHS)
    crit  = nn.CrossEntropyLoss()
    train_dl = DataLoader(TensorDataset(Xw_bal, yw_bal), batch_size=CFG.BATCH_SIZE, shuffle=True)
    val_dl   = DataLoader(TensorDataset(Xw_va, yw_va), batch_size=1024)

    best_f1, best_state, patience = -1.0, None, CFG.PATIENCE
    for epoch in range(1, CFG.MAX_EPOCHS + 1):
        model.train(); tot = 0.0
        for xb, yb in train_dl:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad(); loss = crit(model(xb), yb); loss.backward(); opt.step()
            tot += loss.item()
        sched.step()
        yv, pv, _ = evaluate(model, val_dl)
        vf1 = f1_score(yv, pv)
        print(f'  Epoch {epoch:02d} | loss {tot/len(train_dl):.4f} | val F1 {vf1:.4f} | pred-APT {pv.mean():.3f}')
        if vf1 > best_f1:
            best_f1, patience = vf1, CFG.PATIENCE
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            patience -= 1
            if patience == 0: print(f'  Early stop (best val F1 {best_f1:.4f})'); break
    model.load_state_dict(best_state)

    # --- test trên DỮ LIỆU THẬT ---
    test_dl = DataLoader(TensorDataset(Xw_te, yw_te), batch_size=1024)
    y, p, prob = evaluate(model, test_dl)
    m = {'seed': seed, 'use_acfd': use_acfd,
         'accuracy_%': accuracy_score(y, p) * 100,
         'precision': precision_score(y, p, zero_division=0),
         'recall': recall_score(y, p, zero_division=0),
         'f1': f1_score(y, p, zero_division=0),
         'auc_roc': roc_auc_score(y, prob)}
    print(f'  TEST: ' + ' | '.join(f'{k}={v:.4f}' for k, v in m.items() if isinstance(v, float)))
    print(confusion_matrix(y, p))

    arts = None
    if keep_artifacts:
        arts = {'model': model, 'acfd': acfd, 'scaler': scaler,
                'Xw_tr': Xw_tr, 'yw_tr': yw_tr, 'Xw_te': Xw_te, 'yw_te': yw_te}
    else:
        del model; torch.cuda.empty_cache()
    return m, arts

In [ ]:
# ===== Cell 5: Chạy chính — multi-seed, With ACFD (Table 4 dòng Longformer) =====
results, ART = [], None
for i, seed in enumerate(CFG.SEEDS):
    m, a = run_experiment(seed, use_acfd=True, keep_artifacts=(i == 0))
    results.append(m)
    if a is not None: ART = a

keys = ['accuracy_%', 'precision', 'recall', 'f1', 'auc_roc']
print('\n' + '=' * 64)
print(f'KẾT QUẢ {len(CFG.SEEDS)} SEEDS — ACFD-Transformer (Longformer), params {N_PARAMS/1e6:.2f}M')
print('=' * 64)
summary = {}
for k in keys:
    vals = np.array([r[k] for r in results])
    summary[k] = {'mean': float(vals.mean()), 'std': float(vals.std(ddof=1) if len(vals) > 1 else 0)}
    print(f'  {k:12s}: {vals.mean():.4f} ± {vals.std(ddof=1) if len(vals)>1 else 0:.4f}   (các seed: {np.round(vals,4)})')

with open(os.path.join(CFG.OUT_DIR, 'table4_longformer_real.json'), 'w') as f:
    json.dump({'per_seed': results, 'summary': summary, 'params_M': N_PARAMS/1e6,
               'n_features': len(FEATURES), 'features': FEATURES,
               'real_benign': int((y_all==0).sum()), 'real_attack': int((y_all==1).sum())}, f, indent=2)
torch.save(ART['model'].state_dict(), os.path.join(CFG.OUT_DIR, 'acfd_longformer_seed42.pth'))
print('\n✓ Đã lưu kết quả + model vào', CFG.OUT_DIR)
print('→ Đây là các con số để cập nhật Table 4 (kèm mean ± std).')

In [ ]:
# ===== Cell 6: SHAP THẬT → thay Fig. 4 =====
import shap

model = ART['model'].eval()
bg_idx = torch.randperm(len(ART['Xw_tr']))[:100]
te_idx = torch.randperm(len(ART['Xw_te']))[:300]
background = ART['Xw_tr'][bg_idx].to(device)
explain_x  = ART['Xw_te'][te_idx].to(device)

print('Chạy GradientExplainer (vài phút)...')
explainer = shap.GradientExplainer(model, background)
sv = explainer.shap_values(explain_x)
sv = sv[1] if isinstance(sv, list) else np.array(sv)[..., 1]   # lớp APT
feat_imp = np.abs(np.array(sv)).mean(axis=(0, 1))               # mean |SHAP| qua samples & timesteps

order = np.argsort(feat_imp)[::-1][:15]
plt.figure(figsize=(10, 7))
plt.barh(range(len(order)), feat_imp[order][::-1], color='steelblue')
plt.yticks(range(len(order)), [FEATURES[i] for i in order][::-1])
plt.xlabel('Mean |SHAP value| (APT class)')
plt.title('Feature Importance — SHAP (GradientExplainer, real test data)')
plt.tight_layout()
for ext in ('png', 'pdf'):
    plt.savefig(os.path.join(CFG.OUT_DIR, f'fig4_shap_real.{ext}'), dpi=300, bbox_inches='tight')
plt.show()

with open(os.path.join(CFG.OUT_DIR, 'fig4_shap_values.json'), 'w') as f:
    json.dump({FEATURES[i]: float(feat_imp[i]) for i in order}, f, indent=2)
print('✓ Fig. 4 mới (SHAP thật) đã lưu. Cập nhật cả §5.3 theo các feature thực sự quan trọng ở trên.')

In [ ]:
# ===== Cell 7: Fig. 2 THẬT — original / noised / synthetic trên dữ liệu thật =====
acfd = ART['acfd']
mask = ART['yw_tr'] == 1
x_dim = CFG.WINDOW * len(FEATURES)
real_attack = ART['Xw_tr'][mask].reshape(-1, x_dim)

# chọn feature có |SHAP| cao nhất để minh hoạ (ở timestep cuối)
fidx = int(np.argsort(feat_imp)[::-1][0])
col = (CFG.WINDOW - 1) * len(FEATURES) + fidx

orig = real_attack[:, col].numpy()
t_mid = CFG.T_STEPS // 2
ab = acfd.abar[t_mid].item()
noised = math.sqrt(ab) * orig + math.sqrt(1 - ab) * np.random.randn(len(orig))
n_syn = min(2000, len(orig) * 4)
synth = acfd.sample(n_syn, torch.zeros(1, dtype=torch.long) + 1, x_dim)[:, col].numpy()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, data, title, color in zip(
        axes, [orig, noised, synth],
        [f'(a) Original Attack Data — {FEATURES[fidx]}', f'(b) Noised (t={t_mid}/{CFG.T_STEPS})', '(c) ACFD Synthetic'],
        ['#2E86AB', '#A23B72', '#F18F01']):
    ax.hist(data, bins=40, alpha=0.8, color=color, edgecolor='black')
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xlabel('Feature value'); ax.set_ylabel('Frequency')
plt.suptitle('ACFD Diffusion Process (REAL data)', fontweight='bold')
plt.tight_layout()
for ext in ('png', 'pdf'):
    plt.savefig(os.path.join(CFG.OUT_DIR, f'fig2_diffusion_real.{ext}'), dpi=300, bbox_inches='tight')
plt.show()
print('✓ Fig. 2 mới (dữ liệu thật) đã lưu.')

In [ ]:
# ===== Cell 8: Ablation — Without ACFD (Table 5 thật) =====
abl, _ = run_experiment(CFG.SEEDS[0], use_acfd=False)

print('\n' + '=' * 64)
print('TABLE 5 (THẬT) — Impact of ACFD module')
print('=' * 64)
with_acfd = results[0]   # cùng seed
print(f"{'Configuration':32s} {'Acc.(%)':>8s} {'Prec.':>7s} {'Rec.':>7s} {'F1':>7s}")
print(f"{'Without ACFD (Imbalanced)':32s} {abl['accuracy_%']:8.2f} {abl['precision']:7.4f} {abl['recall']:7.4f} {abl['f1']:7.4f}")
print(f"{'With ACFD (Balanced)':32s} {with_acfd['accuracy_%']:8.2f} {with_acfd['precision']:7.4f} {with_acfd['recall']:7.4f} {with_acfd['f1']:7.4f}")

with open(os.path.join(CFG.OUT_DIR, 'table5_ablation_real.json'), 'w') as f:
    json.dump({'without_acfd': abl, 'with_acfd': with_acfd}, f, indent=2)
print('\n✓ Đã lưu. Nếu ACFD không cải thiện → báo cáo trung thực và thảo luận trong §6.')

## Checklist cập nhật bản thảo sau khi chạy xong

1. **Table 1**: thay bằng số THẬT in ở Cell 2 (benign sample / attack thật / 63 features); ghi rõ quy trình sample benign và rằng synthetic CHỈ dùng trong train, không tính vào thống kê dataset.
2. **Table 3**: giữ T=1000 (giờ đã chạy đúng); sửa mô tả attention window (full attention trên W=10); thêm hardware thật (Colab Pro, GPU gì in ở Cell 1).
3. **Table 4**: dòng Longformer = mean ± std từ Cell 5; Params = số in ở Cell 3. Các variant khác (Performer/Linformer/Hierarchical) nếu giữ thì phải chạy lại tương tự — nếu không kịp, thu hẹp bảng về Longformer vs Standard Transformer và sửa claim trong abstract/§1.3.
4. **Table 5**: từ Cell 8.
5. **Fig. 2**: thay bằng `fig2_diffusion_real.pdf`. **Fig. 4**: thay bằng `fig4_shap_real.pdf` và viết lại §5.3 theo feature thực.
6. **§5.4 Error Analysis**: làm lại từ confusion matrix thật.
7. **Repo GitHub**: cập nhật code khớp notebook này + hướng dẫn tải data → trả lời reviewer "code không chạy được".
8. Mọi con số trong bản thảo phải truy được về một file json/cell output trong `acfd_v3_outputs/`.
